# Decision Trees
Companion to the decision tree notes.

## 1. Gini and entropy for nodes of different purity

In [ ]:
import numpy as np
for ones in range(7, -1, -1):
    p0 = 8 / (8 + ones); p1 = 1 - p0
    entropy = -sum(p * np.log2(p) for p in (p0, p1) if p > 0)
    gini = 1 - p0**2 - p1**2
    print(f"8 zeros, {ones} ones: entropy {entropy:.3f}  gini {gini:.3f}")

## 2. The six-point classification split, every candidate

In [ ]:
x = np.array([1.2, 2.4, 3.1, 4.5, 6.0, 7.2]); y = np.array([0, 0, 1, 0, 1, 1])
gini = lambda v: 1 - sum(np.mean(v == k)**2 for k in (0, 1)) if len(v) else 0
for t in (x[:-1] + x[1:]) / 2:
    L, R = y[x <= t], y[x > t]
    w = (len(L) * gini(L) + len(R) * gini(R)) / len(y)
    print(f"x <= {t:.2f}: weighted gini {w:.4f}  reduction {gini(y) - w:.4f}")

## 3. The regression split by hand

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6.]); y = np.array([10, 12, 11, 30, 32, 31.])
mse = lambda v: ((v - v.mean())**2).mean() if len(v) else 0
for t in (x[:-1] + x[1:]) / 2:
    L, R = y[x <= t], y[x > t]
    w = (len(L) * mse(L) + len(R) * mse(R)) / len(y)
    print(f"x <= {t}: weighted MSE {w:7.3f}  reduction {mse(y) - w:7.3f}")

## 4. Bank data: an unpruned tree

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

bd = pd.read_csv("../../data/bd_train.csv").drop(columns=["REF_NO", "post_code", "post_area"])
y = (bd.pop("Revenue.Grid") == 1).astype(int)
X = pd.get_dummies(bd, drop_first=True, dtype=float)       # no scaling needed for trees
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

full = DecisionTreeClassifier(random_state=42).fit(x_train, y_train)
print("unpruned: leaves", full.get_n_leaves(), " depth", full.get_depth(),
      " train AUC", round(roc_auc_score(y_train, full.predict_proba(x_train)[:, 1]), 3),
      " test AUC", round(roc_auc_score(y_test, full.predict_proba(x_test)[:, 1]), 3))

## 5. Cost-complexity pruning by cross-validation

In [ ]:
path = full.cost_complexity_pruning_path(x_train, y_train)
alphas = np.unique(np.quantile(path.ccp_alphas, np.linspace(0, 0.99, 30)))   # 30 candidates spread over the path
cv_auc = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=42), x_train, y_train,
                          cv=5, scoring="roc_auc").mean() for a in alphas]
best_alpha = alphas[int(np.argmax(cv_auc))]
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42).fit(x_train, y_train)
print(f"best ccp_alpha {best_alpha:.5f}: leaves {pruned.get_n_leaves()}, depth {pruned.get_depth()},"
      f" CV AUC {max(cv_auc):.3f}, test AUC {roc_auc_score(y_test, pruned.predict_proba(x_test)[:, 1]):.3f}")

## 6. A small tree you can read

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree, export_text
small = DecisionTreeClassifier(max_depth=2, random_state=0).fit(x_train, y_train)
print(export_text(small, feature_names=list(x_train.columns)))
plt.figure(figsize=(12, 5)); plot_tree(small, feature_names=list(x_train.columns), filled=True, fontsize=8)

## 7. Loan data: regression trees of different depths

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

ld = pd.read_csv("../../data/loan_data_train.csv")
ld["Interest.Rate"] = pd.to_numeric(ld["Interest.Rate"].str.replace("%", ""), errors="coerce")
ld["FICO"] = ld["FICO.Range"].str.split("-", expand=True).astype(float).mean(axis=1)
ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
for col in ["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")
cols = ["FICO", "Loan.Length", "Amount.Requested", "Inquiries.in.the.Last.6.Months",
        "Monthly.Income", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]
ld = ld[cols + ["Interest.Rate"]].dropna()
xr_train, xr_test, yr_train, yr_test = train_test_split(ld[cols], ld["Interest.Rate"], test_size=0.2, random_state=2)
for depth in [2, 4, 6, 10, None]:
    t = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(xr_train, yr_train)
    print(f"max_depth={str(depth):>4}: leaves {t.get_n_leaves():4d}  train MAE {mean_absolute_error(yr_train, t.predict(xr_train)):.3f}"
          f"  test MAE {mean_absolute_error(yr_test, t.predict(xr_test)):.3f}")

## 8. Instability: trees from slightly different samples

In [ ]:
rng = np.random.default_rng(0)
for trial in range(3):
    idx = rng.choice(len(x_train), size=len(x_train), replace=True)       # a slightly different training sample
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(x_train.iloc[idx], y_train.iloc[idx])
    feats = [x_train.columns[f] for f in t.tree_.feature if f >= 0]
    print(f"sample {trial + 1}: splits on {feats}")

## Try it yourself
1. Entropy-based reduction for the splits at 2.75 and 3.80 in section 2.
2. Add (7, 12) to the regression example and find the best split.
3. `min_samples_leaf` of 1, 20, 100 on the bank tree.
4. Depth-6 loan tree with an extra `log1p(Monthly.Income)` column: any change? Why (not)?